# 03 - Baseline Out-of-Fold Predictions

This notebook generates leakage-safe **out-of-fold (OOF) predictions** for the selected baseline model:

> **Logistic Regression - Baseline v1**

Notebook 02 compared Logistic Regression, Random Forest, and XGBoost using five-fold `StratifiedGroupKFold`. Logistic Regression was retained because it showed comparatively stable validation behavior, smaller train–validation gaps, and lower computational cost.

The purpose of this notebook is narrower:

1. reproduce the same 30% working sample and grouped-validation setup;
2. train a fresh Logistic Regression model in each fold;
3. score only the unseen validation rows from that fold;
4. preserve drive identity and observation date;
5. combine all validation scores into one OOF dataset.

No classification threshold is selected here. Threshold selection, drive-level aggregation, and lead-time analysis belong to later notebooks.


## Why out-of-fold predictions are needed

A prediction is **out of fold** when it comes from a model that did not train on that observation.

With five folds:

- four folds are used for training;
- the remaining fold is used for validation;
- the process repeats until every row has been held out once.

Because the split is grouped by `serial_number`, all sampled observations from a physical drive remain together. A drive scored in a validation fold is therefore unseen by that fold's model.

OOF scores are more appropriate than in-sample scores for downstream threshold analysis because they better represent model behavior on unseen drives.


In [ ]:
from __future__ import annotations

from pathlib import Path
import gc
import time

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## 1. Configuration

The settings below match the authoritative grouped-validation experiment in Notebook 02.

The 30% sample is a reproducible **row sample of drive-day observations**. It is not a sample of complete drive histories. This choice is retained for consistency with the completed model comparison.


In [ ]:
DATA_PATH = Path(
    "/kaggle/input/datasets/emilywanglovescats/"
    "blackblaze-data-2023-cleaned/df_cleaned.parquet"
)

OUTPUT_DIR = Path("/kaggle/working")
OOF_OUTPUT_PATH = OUTPUT_DIR / "logistic_oof_with_drive_info.parquet"

TARGET_COLUMN = "failure"
GROUP_COLUMN = "serial_number"
DATE_COLUMN = "date"

FEATURE_COLUMNS = [
    "smart_5_raw",
    "smart_197_raw",
    "smart_198_raw",
]

SAMPLE_FRACTION = 0.30
N_SPLITS = 5
RANDOM_STATE = 42


## 2. Load the cleaned dataset

Only the columns required for modeling and downstream drive-level analysis are loaded:

- three SMART baseline features;
- the failure target;
- `serial_number` for grouped splitting and later aggregation;
- `date` for later warning lead-time analysis.

The metadata columns are preserved in the OOF output but are not supplied to the model as predictive features.


In [ ]:
def load_data(
    file_path: Path,
    sample_fraction: float,
    random_state: int,
) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Dataset not found at:\n{file_path}\n\n"
            "Update DATA_PATH to the Kaggle location of df_cleaned.parquet."
        )

    if not 0 < sample_fraction <= 1:
        raise ValueError("sample_fraction must be in the interval (0, 1].")

    required_columns = (
        FEATURE_COLUMNS
        + [TARGET_COLUMN, GROUP_COLUMN, DATE_COLUMN]
    )

    df = pd.read_parquet(
        file_path,
        columns=required_columns,
    )

    missing_columns = set(required_columns) - set(df.columns)
    if missing_columns:
        raise ValueError(
            f"Dataset is missing required columns: {sorted(missing_columns)}"
        )

    if sample_fraction < 1.0:
        df = df.sample(
            frac=sample_fraction,
            random_state=random_state,
        ).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    if df[TARGET_COLUMN].nunique() < 2:
        raise ValueError(
            "The working sample contains only one target class. "
            "Increase SAMPLE_FRACTION."
        )

    if df[GROUP_COLUMN].isna().any():
        raise ValueError("serial_number contains missing values.")

    if df[DATE_COLUMN].isna().any():
        raise ValueError("date contains missing values.")

    df[DATE_COLUMN] = pd.to_datetime(
        df[DATE_COLUMN],
        errors="raise",
    )

    return df


df = load_data(
    file_path=DATA_PATH,
    sample_fraction=SAMPLE_FRACTION,
    random_state=RANDOM_STATE,
)

print(f"Working rows: {len(df):,}")
print(f"Unique drives: {df[GROUP_COLUMN].nunique():,}")
print(f"Positive rows: {int(df[TARGET_COLUMN].sum()):,}")
print(f"Failure prevalence: {df[TARGET_COLUMN].mean():.6%}")
print(
    "Date range: "
    f"{df[DATE_COLUMN].min().date()} to "
    f"{df[DATE_COLUMN].max().date()}"
)
print(
    "Approximate memory usage: "
    f"{df.memory_usage(deep=True).sum() / 1e9:.2f} GB"
)

display(df.head())


## 3. Separate features, target, groups, and metadata

`serial_number` is used as the grouping key, not as a model feature. This prevents the model from learning drive identity.

The dataframe index is reset during loading. Fold indices produced by scikit-learn can therefore be used safely as positional indices with `.iloc`.

OOF probability and fold arrays are preallocated to the exact length of the working dataset. Each validation prediction will be written back to its original sampled-row position. This protects row alignment when fold results are combined.


In [ ]:
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].astype("int8").copy()
groups = df[GROUP_COLUMN].copy()

serial_numbers = df[GROUP_COLUMN].copy()
dates = df[DATE_COLUMN].copy()

oof_probabilities = np.full(len(df), np.nan, dtype=np.float64)
oof_folds = np.full(len(df), -1, dtype=np.int16)

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Group vector shape: {groups.shape}")


## 4. Define the baseline pipeline

The pipeline is unchanged from Notebook 02:

1. median imputation;
2. standardization;
3. class-weighted Logistic Regression.

A fresh pipeline is created for every fold. Preprocessing is fitted only on the training partition, preventing validation information from leaking into imputation or scaling.


In [ ]:
def create_logistic_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


## 5. Generate five-fold OOF predictions

`StratifiedGroupKFold` attempts to balance the rare failure class while keeping every sampled row from the same drive in one fold.

For each fold, the notebook:

- verifies zero serial-number overlap;
- trains a fresh Logistic Regression pipeline;
- predicts probabilities for the held-out rows only;
- writes predictions and fold labels back to their original row positions;
- records fold-level diagnostics.

No probability threshold is applied.


In [ ]:
splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

fold_records: list[dict[str, int | float]] = []

for fold, (train_index, valid_index) in enumerate(
    splitter.split(X, y, groups=groups)
):
    X_train = X.iloc[train_index]
    X_valid = X.iloc[valid_index]
    y_train = y.iloc[train_index]
    y_valid = y.iloc[valid_index]

    train_drives = set(groups.iloc[train_index])
    valid_drives = set(groups.iloc[valid_index])
    overlap = train_drives.intersection(valid_drives)

    if overlap:
        raise RuntimeError(
            f"Leakage detected in Fold {fold}: "
            f"{len(overlap)} serial numbers appear in both partitions."
        )

    if np.any(oof_folds[valid_index] != -1):
        raise RuntimeError(
            f"Fold {fold} attempts to overwrite previously assigned OOF rows."
        )

    print("\n" + "=" * 78)
    print(f"FOLD {fold + 1}/{N_SPLITS}")
    print("=" * 78)
    print(f"Training rows: {len(train_index):,}")
    print(f"Validation rows: {len(valid_index):,}")
    print(f"Training drives: {len(train_drives):,}")
    print(f"Validation drives: {len(valid_drives):,}")
    print(f"Training positives: {int(y_train.sum()):,}")
    print(f"Validation positives: {int(y_valid.sum()):,}")
    print(f"Serial-number overlap: {len(overlap)}")

    model = create_logistic_pipeline()

    start_time = time.perf_counter()
    model.fit(X_train, y_train)
    validation_probabilities = model.predict_proba(X_valid)[:, 1]
    fit_seconds = time.perf_counter() - start_time

    if len(validation_probabilities) != len(valid_index):
        raise RuntimeError(
            f"Fold {fold} prediction count does not match validation rows."
        )

    oof_probabilities[valid_index] = validation_probabilities
    oof_folds[valid_index] = fold

    fold_records.append(
        {
            "fold": fold,
            "train_rows": len(train_index),
            "validation_rows": len(valid_index),
            "train_drives": len(train_drives),
            "validation_drives": len(valid_drives),
            "train_positives": int(y_train.sum()),
            "validation_positives": int(y_valid.sum()),
            "train_failure_rate": float(y_train.mean()),
            "validation_failure_rate": float(y_valid.mean()),
            "serial_overlap": len(overlap),
            "fit_seconds": fit_seconds,
        }
    )

    print(f"Fit and prediction time: {fit_seconds:.2f} seconds")
    print(f"Stored OOF predictions: {len(valid_index):,}")

    del (
        X_train,
        X_valid,
        y_train,
        y_valid,
        model,
        validation_probabilities,
    )
    gc.collect()


## 6. Review fold coverage

Every sampled row must be assigned to exactly one validation fold. Every sampled drive must also belong to only one validation fold.

These checks validate the OOF assembly process; they do not measure predictive performance.


In [ ]:
fold_diagnostics = pd.DataFrame(fold_records)

display(
    fold_diagnostics[
        [
            "fold",
            "train_rows",
            "validation_rows",
            "train_drives",
            "validation_drives",
            "validation_positives",
            "validation_failure_rate",
            "serial_overlap",
            "fit_seconds",
        ]
    ]
)

if np.isnan(oof_probabilities).any():
    missing_count = int(np.isnan(oof_probabilities).sum())
    raise RuntimeError(
        f"{missing_count:,} rows did not receive an OOF probability."
    )

if np.any(oof_folds < 0):
    unassigned_count = int((oof_folds < 0).sum())
    raise RuntimeError(
        f"{unassigned_count:,} rows did not receive a fold assignment."
    )

if set(np.unique(oof_folds)) != set(range(N_SPLITS)):
    raise RuntimeError("The OOF arrays do not contain all expected folds.")

print("Every sampled row received exactly one OOF prediction.")
print(f"Assigned folds: {sorted(np.unique(oof_folds).tolist())}")


## 7. Assemble the OOF dataset

The output retains the minimum fields required for later analysis:

| Column | Meaning |
|---|---|
| `fold` | Held-out fold that produced the prediction |
| `serial_number` | Physical-drive identifier |
| `date` | Date of the drive-day observation |
| `y_true` | Observed failure label |
| `y_prob` | Leakage-safe Logistic Regression probability |

The output is kept in the original order of the reproducibly sampled dataframe. The original row index is not exported because it has no downstream analytical role.


In [ ]:
oof_predictions = pd.DataFrame(
    {
        "fold": oof_folds,
        "serial_number": serial_numbers.to_numpy(),
        "date": dates.to_numpy(),
        "y_true": y.to_numpy(),
        "y_prob": oof_probabilities,
    }
)

display(oof_predictions.head())
print(f"OOF rows: {len(oof_predictions):,}")
print(f"Unique drives: {oof_predictions['serial_number'].nunique():,}")
print(f"Positive rows: {int(oof_predictions['y_true'].sum()):,}")


## 8. Final integrity checks

The checks below confirm:

- row-count preservation;
- complete fold coverage;
- valid target and probability ranges;
- nonmissing metadata;
- target alignment with the working sample;
- one validation fold per drive.

The last condition is especially important: a physical drive must never receive OOF predictions from multiple validation folds.


In [ ]:
expected_columns = [
    "fold",
    "serial_number",
    "date",
    "y_true",
    "y_prob",
]

if list(oof_predictions.columns) != expected_columns:
    raise RuntimeError("Unexpected OOF output schema.")

if len(oof_predictions) != len(df):
    raise RuntimeError(
        "OOF row count does not match the working dataset."
    )

if oof_predictions["fold"].nunique() != N_SPLITS:
    raise RuntimeError("Not all folds are present in the OOF output.")

if not oof_predictions["y_true"].isin([0, 1]).all():
    raise RuntimeError("Unexpected values found in y_true.")

if not oof_predictions["y_prob"].between(0, 1).all():
    raise RuntimeError("Some OOF probabilities fall outside [0, 1].")

if oof_predictions[["serial_number", "date"]].isna().any().any():
    raise RuntimeError("Missing drive metadata found in the OOF output.")

if not np.array_equal(
    oof_predictions["y_true"].to_numpy(),
    y.to_numpy(),
):
    raise RuntimeError("Target alignment was not preserved.")

folds_per_drive = (
    oof_predictions.groupby("serial_number")["fold"].nunique()
)

if folds_per_drive.max() != 1:
    affected_drives = int((folds_per_drive > 1).sum())
    raise RuntimeError(
        f"{affected_drives:,} drives appear in more than one validation fold."
    )

print("All OOF integrity checks passed.")
print("Maximum validation folds per drive:", int(folds_per_drive.max()))
print(
    "Probability range: "
    f"{oof_predictions['y_prob'].min():.6f} to "
    f"{oof_predictions['y_prob'].max():.6f}"
)


## 9. Save the OOF predictions

The Parquet file is written to Kaggle's working directory.

This file is an intermediate analytical artifact and should not be committed to the public GitHub repository because of its size and drive-level identifiers. Later public notebooks can document its schema and provide compact summary outputs instead.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

oof_predictions.to_parquet(
    OOF_OUTPUT_PATH,
    index=False,
)

saved_check = pd.read_parquet(OOF_OUTPUT_PATH)

if len(saved_check) != len(oof_predictions):
    raise RuntimeError("Saved Parquet row count does not match memory.")

if list(saved_check.columns) != expected_columns:
    raise RuntimeError("Saved Parquet schema does not match the expected schema.")

print(f"Saved: {OOF_OUTPUT_PATH}")
print(f"File size: {OOF_OUTPUT_PATH.stat().st_size / 1e6:.2f} MB")
print(f"Saved rows: {len(saved_check):,}")
print(f"Saved columns: {list(saved_check.columns)}")


## Conclusion

This notebook generated one leakage-safe Logistic Regression probability for every row in the reproducible working sample.

Each score came from a model that:

- did not train on that row;
- did not train on any sampled row from the same `serial_number`;
- fitted its imputer and scaler using training data only.

The resulting `logistic_oof_with_drive_info.parquet` preserves the drive and date metadata required for the next stage.

The next notebook will convert these row-level OOF scores into drive-level alert behavior and evaluate operational tradeoffs such as alert workload, failed-drive recall, precision, and warning lead time.
